In [3]:
# !pip install pandas numpy scikit-learn yfinance matplotlib

In [4]:
import requests
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import warnings
warnings.filterwarnings("ignore")

In [132]:
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.options.display.float_format = '{:,.3f}'.format
pd.set_option('display.max_columns', None)
# ширина каждого столбца (в символах)
pd.set_option('display.max_colwidth', None)

In [4]:
# Создаём объект сессии для выполнения HTTP-запросов
SESSION = requests.Session()

# Настраиваем стратегию повторных попыток на случай временных сбоев
retries = Retry(
    total=10, # Максимальное количество попыток (включая первую)
    backoff_factor=0.5,  # Задержка = {backoff_factor} * (2 ** (попытка - 1))
    status_forcelist=[500, 502, 503, 504], # Список HTTP-статусов, при которых нужно повторить запрос
    allowed_methods=["GET"] # Разрешаем повтор только для идемпотентных методов (GET).
)

# Монтируем HTTPAdapter к сессии для протокола HTTPS.
# Адаптер отвечает за фактическую отправку запросов и применяет
# настроенную стратегию повторных попыток ко всем HTTPS-запросам через эту сессию.
SESSION.mount("https://", HTTPAdapter(max_retries=retries))


# Получаем “универс” — акции 1-го котировального списка

In [5]:
def get_first_list_tickers() -> list[str]:
    """
    Получает список акций первого котировального списка Московской биржи (MOEX),
    используя основной рынок акций TQBR.

    TQBR — это основной режим торгов акциями на MOEX, на котором обращаются только
    реальные обыкновенные и привилегированные акции с наибольшей ликвидностью.
    Использование данного рынка гарантирует исключение ETF, БПИФов, облигаций,
    структурных нот и прочих неакционных инструментов.

    Сформированный список тикеров является торговым универсумом модели.
    """

    # Эндпоинт MOEX ISS, содержащий справочник всех акций,
    # допущенных к основному режиму торгов TQBR
    url = "https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities.json"

    # Выполняем HTTP-запрос к официальному API Московской биржи
    # Параметр iss.meta=off отключает служебную мета-информацию и уменьшает размер ответа
    r = requests.get(url, params={"iss.meta": "off"})
    r.raise_for_status()   # Прерываем выполнение в случае сетевой или серверной ошибки

    # MOEX возвращает данные в формате: columns + data.
    # Восстанавливаем таблицу со справочной информацией по всем акциям.
    cols = r.json()["securities"]["columns"]
    data = r.json()["securities"]["data"]
    df = pd.DataFrame(data, columns=cols)

    # LISTLEVEL = 1 — первый котировальный список.
    # Это наиболее ликвидные и капитализированные акции (blue chips),
    # которые формируют стабильный и реалистичный торговый универсум.
    df = df[df["LISTLEVEL"] == 1]

    # SECID — биржевой тикер акции (например: SBER, GAZP, LKOH).
    # Удаляем возможные пропуски, берём уникальные значения и сортируем список
    # для обеспечения воспроизводимости экспериментов.
    tickers = sorted(df["SECID"].dropna().unique())

    return tickers


# Формируем торговый универсум для обучения универсальной ML-модели
tickers = get_first_list_tickers()

# Контрольная проверка: количество и первые элементы списка тикеров
len(tickers), tickers[:10]


(69,
 ['AFKS',
  'AFLT',
  'ALRS',
  'AQUA',
  'ASTR',
  'BELU',
  'BSPB',
  'CBOM',
  'CHMF',
  'CNRU'])

# Скачиваем историю свечей по MOEX ISS (с пагинацией)

MOEX выдаёт историю “порциями”, поэтому нужен параметр start (смещение).

Мы будем брать: TRADEDATE, OPEN, HIGH, LOW, CLOSE, VOLUME и собирать в единый “panel dataset”.

In [11]:
def load_moex_history_one(ticker, start_date="2014-01-01", end_date=None, board="TQBR"):
    """
    Загружает полную историческую дневную OHLCV-серию по одной акции с официального API MOEX.

    Функция автоматически обходит ограничение MOEX на размер ответа (100 строк на запрос)
    за счёт постраничной загрузки данных (pagination) и объединяет все страницы
    в единую временную серию.

    Параметры
    ----------
    ticker : str
        Биржевой тикер акции (например: 'SBER', 'GAZP', 'LKOH').

    start_date : str, default='2014-01-01'
        Начальная дата исторической выборки в формате 'YYYY-MM-DD'.

    end_date : str | None, default=None
        Конечная дата выборки. Если None — данные загружаются по последний торговый день.

    board : str, default='TQBR'
        Основной режим торгов MOEX. 'TQBR' используется для получения наиболее ликвидных
        и корректных рыночных котировок.
    """

    # Исторический эндпоинт MOEX для рынка акций
    url = f"https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/{board}/securities/{ticker}.json"

    all_rows = []   # Сюда будут накапливаться все страницы истории
    start = 0       # Смещение для постраничной загрузки (pagination)

    # MOEX выдаёт не более 100 строк за один запрос,
    # поэтому история загружается в цикле с постепенным смещением "start"
    while True:
        r = SESSION.get(
            url,
            params={
                "from": start_date,     # дата начала выборки
                "till": end_date,       # дата окончания выборки
                "start": start,         # смещение страницы
                "iss.meta": "off",      # отключаем служебную мета-информацию
            },
            timeout=50
        )

        # Если сервер вернул ошибку — прекращаем загрузку для данного тикера
        if r.status_code != 200:
            break

        js = r.json()
        cols = js["history"]["columns"]
        data = js["history"]["data"]

        # Если сервер не вернул данных — история закончилась
        if not data:
            break

        # Преобразуем текущую страницу в DataFrame и добавляем в общий список
        all_rows.append(pd.DataFrame(data, columns=cols))

        # MOEX всегда отдаёт максимум 100 строк.
        # Если получено меньше — это последняя страница истории.
        if len(data) < 100:
            break

        # Смещаемся на следующую страницу
        start += 100

        # Небольшая задержка между запросами — обязательна,
        # чтобы избежать серверных ограничений и обрывов соединения
        time.sleep(0.2)

    # Если MOEX не вернул ни одной страницы — возвращаем пустой DataFrame
    if not all_rows:
        return pd.DataFrame()

    # Объединяем все страницы истории в единую временную серию
    df = pd.concat(all_rows)

    # Оставляем только финансово значимые поля OHLCV
    df = df[["TRADEDATE", "OPEN", "HIGH", "LOW", "CLOSE", "VOLUME"]]
    df.columns = [c.lower() for c in df.columns]

    # Приводим дату к формату datetime и сортируем временной ряд
    df["tradedate"] = pd.to_datetime(df["tradedate"])
    df = df.sort_values("tradedate").set_index("tradedate")

    # Добавляем тикер как признак для формирования panel-датасета
    df["ticker"] = ticker

    return df


In [12]:
def load_moex_universe(
    tickers: list[str],
    start_date: str = "2014-01-01",
    end_date: str | None = None,
) -> pd.DataFrame:
    """
    Загружает исторические котировки для всех акций торгового универсума MOEX
    и формирует единый panel-датасет для последующего машинного обучения.

    Каждая акция загружается отдельно через load_moex_history_one(), после чего
    все временные ряды объединяются в одну таблицу формата:

    [date, open, high, low, close, volume, ticker]

    Такой формат называется panel data и является стандартом для финансового ML.


    Параметры
    ----------
    tickers : list[str]
        Список биржевых тикеров акций, формирующих торговый универсум
        (например: ['SBER', 'GAZP', 'LKOH', ...]).

    start_date : str, default='2014-01-01'
        Начальная дата выборки в формате 'YYYY-MM-DD'.
        Определяет начало исторического периода, используемого для обучения модели.

    end_date : str | None, default=None
        Конечная дата выборки в формате 'YYYY-MM-DD'.
        Если None — данные загружаются по последнюю доступную торговую сессию.

    Возвращает
    ----------
    pd.DataFrame
        Единый panel-датасет формата:
        [date, open, high, low, close, volume, ticker],
        где каждая строка соответствует одной акции в один торговый день.
    """

    frames = []   # сюда будут собираться DataFrame'ы с котировками по каждой акции
    bad = []      # список тикеров, по которым не удалось получить корректные данные

    # Проходим по всем акциям торгового универсума
    for t in tqdm(tickers, desc="Loading MOEX universe"):
        try:
            # Загружаем исторические данные по одной акции
            dft = load_moex_history_one(
                        t,                     # тикер акции (например: 'SBER', 'GAZP', 'LKOH')
                        start_date=start_date, # дата начала исторической выборки
                        end_date=end_date,     # дата окончания (None = по последний торговый день)
                        board = "TQBR"         # режим торгов
                    )
            time.sleep(0.3)
            # Иногда MOEX возвращает пустой набор (делистинг, технические бумаги и т.п.)
            # Такие акции исключаем из датасета, чтобы не загрязнять модель шумом
            if dft.empty:
                print('bad')
                bad.append(t)
                continue
    
            frames.append(dft)

        # В случае любой сетевой или серверной ошибки просто исключаем тикер
        # и продолжаем сбор данных по остальным акциям
        except Exception:
            bad.append(t)

    # Объединяем котировки всех акций в один большой panel dataset
    panel = pd.concat(frames, axis=0).reset_index().rename(columns={"tradedate": "date"})

    # Итоговый формат данных:
    # date | open | high | low | close | volume | ticker
    # где каждая строка — один торговый день по одной акции
    return panel, bad


# Формируем сырой датасет по всему рынку
df, skipped = load_moex_universe(tickers, start_date="2010-01-01")

# Первичный осмотр данных
df.head()


Loading MOEX universe: 100%|██████████| 69/69 [07:01<00:00,  6.10s/it]


,date,open,high,low,close,volume,ticker
0,2014-06-09,44.364,45.001,43.751,44.448,4380200,AFKS
1,2014-06-10,44.440,45.596,44.117,45.499,11586400,AFKS
2,2014-06-11,45.007,45.749,44.700,45.300,4757700,AFKS
3,2014-06-16,45.913,46.370,44.514,45.999,17932600,AFKS
4,2014-06-17,46.300,46.467,45.700,46.100,5544500,AFKS


In [13]:
path = f'prepared_data/dataset_2010_01_01.pickle'

with open(path, 'wb') as f:
    pickle.dump(df, f)

In [14]:
skipped

[]

In [15]:
df.shape

(146853, 7)

In [16]:
df.ticker.unique()

array(['AFKS', 'AFLT', 'ALRS', 'AQUA', 'ASTR', 'BELU', 'BSPB', 'CBOM',
       'CHMF', 'CNRU', 'DOMRF', 'ELFV', 'ENPG', 'EUTR', 'FEES', 'FIXR',
       'FLOT', 'GAZP', 'GEMC', 'GMKN', 'HEAD', 'HYDR', 'IRAO', 'LEAS',
       'LENT', 'LKOH', 'LSRG', 'MAGN', 'MBNK', 'MDMG', 'MOEX', 'MSNG',
       'MTLR', 'MTLRP', 'MTSS', 'MVID', 'NLMK', 'NVTK', 'OZON', 'OZPH',
       'PHOR', 'PIKK', 'PLZL', 'POSI', 'RAGR', 'RENI', 'RNFT', 'ROSN',
       'RTKM', 'RTKMP', 'RUAL', 'SBER', 'SBERP', 'SELG', 'SFIN', 'SGZH',
       'SMLT', 'SVCB', 'T', 'TATN', 'TATNP', 'TGKA', 'TRNFP', 'UPRO',
       'VKCO', 'VSEH', 'VTBR', 'X5', 'YDEX'], dtype=object)

In [17]:
df.shape[0]/69

2128.304347826087

In [20]:
df[df['ticker'] =='SBER'].sort_values(by=['date']).head(10)

,date,open,high,low,close,volume,ticker
79955,2013-03-25,NaN,NaN,NaN,NaN,0,NVTK
114487,2013-03-25,75.02000,75.05000,73.21000,73.35000,120300,SBERP
46697,2013-03-25,1951.70000,1963.70000,1920.70000,1922.00000,32441,LKOH
111216,2013-03-25,96.00000,101.14000,96.00000,98.79000,593680,SBER
38663,2013-03-25,NaN,NaN,NaN,NaN,0,HYDR
142864,2013-03-25,0.05181,0.05194,0.05038,0.05043,35450700000,VTBR
46698,2013-03-26,1920.50000,1944.50000,1919.80000,1944.50000,3653,LKOH
114488,2013-03-26,72.64000,74.06000,71.34000,72.30000,209100,SBERP
111217,2013-03-26,98.58000,99.31000,97.08000,97.20000,1283550,SBER
79956,2013-03-26,NaN,NaN,NaN,NaN,0,NVTK


In [23]:
df.groupby('ticker').size().sort_values().describe()#.head(10)

count      69.000000
mean     2128.304348
std      1172.791922
min        72.000000
25%      1072.000000
50%      2968.000000
75%      3199.000000
max      3271.000000
dtype: float64

# Обработка ошибочных записей (логические некорректные)

In [97]:
path = f'prepared_data/dataset_2010_01_01.pickle'

with open(path, 'rb') as f:
    df = pickle.load(f)

In [98]:
from collections import defaultdict

audit = defaultdict(int)

In [99]:
# Удаление дублей 
dup_mask = df.duplicated(subset=["date","ticker"])
audit["duplicates"] = dup_mask.sum()
if audit["duplicates"]>0:
    display(df[dup_mask])
df = df[~dup_mask]

In [100]:
# Пропуски OHLC
na_mask = df[["open","high","low","close"]].isna().any(axis=1)
audit["missing_ohlc"] = na_mask.sum()
if audit["missing_ohlc"]>0:
    display(df[na_mask])
df = df[~na_mask]

,date,open,high,low,close,volume,ticker
1913,2022-01-07,NaN,NaN,NaN,NaN,0,AFKS
1946,2022-02-23,NaN,NaN,NaN,NaN,0,AFKS
1949,2022-02-28,NaN,NaN,NaN,NaN,0,AFKS
1950,2022-03-01,NaN,NaN,NaN,NaN,0,AFKS
1951,2022-03-02,NaN,NaN,NaN,NaN,0,AFKS
...,...,...,...,...,...,...,...
146436,2024-07-17,NaN,NaN,NaN,NaN,0,YDEX
146437,2024-07-18,NaN,NaN,NaN,NaN,0,YDEX
146438,2024-07-19,NaN,NaN,NaN,NaN,0,YDEX
146439,2024-07-22,NaN,NaN,NaN,NaN,0,YDEX


In [101]:
# Нулевые/отрицательные цены

bad_price = (df["open"]<=0)|(df["high"]<=0)|(df["low"]<=0)|(df["close"]<=0)
audit["non_positive_price"] = bad_price.sum()
if audit["non_positive_price"]>0:
    display(df[bad_price])
df = df[~bad_price]

In [102]:
# Логические ошибки свечей
logic_mask = (
    (df["high"] < df["low"]) |
    (df["high"] < df["open"]) |
    (df["high"] < df["close"]) |
    (df["low"] > df["open"]) |
    (df["low"] > df["close"])
)
audit["ohlc_logic_error"] = logic_mask.sum()
if audit["ohlc_logic_error"]>0:
    display(df[logic_mask])
df = df[~logic_mask]

In [103]:
# Нулевые объёмы

zero_vol = df["volume"] <= 0
audit["zero_volume"] = zero_vol.sum()
if audit["zero_volume"]>0:
    display(df[zero_vol])
df = df[~zero_vol]

In [104]:
audit

defaultdict(int,
            {'duplicates': np.int64(0),
             'missing_ohlc': np.int64(1953),
             'non_positive_price': np.int64(0),
             'ohlc_logic_error': np.int64(0),
             'zero_volume': np.int64(0)})

# Дробление и обратный сплит обработка таких случаев

In [114]:
# Обратный сплит по VTBR
mask = (df['ticker'] == 'VTBR') & (df['date'] < '2024-07-15')
df.loc[mask, 'open'] = df.loc[mask, 'open'] * 5000
df.loc[mask, 'high'] = df.loc[mask, 'high'] * 5000
df.loc[mask, 'low']  = df.loc[mask, 'low'] * 5000
df.loc[mask, 'close'] = df.loc[mask, 'close'] * 5000
df.loc[mask, 'volume'] = df.loc[mask, 'volume'] / 5000

# Обратный сплит по IRAO
mask = (df['ticker'] == 'IRAO') & (df['date'] < '2015-01-20')
df.loc[mask, 'open'] = df.loc[mask, 'open'] * 100
df.loc[mask, 'high'] = df.loc[mask, 'high'] * 100
df.loc[mask, 'low']  = df.loc[mask, 'low'] * 100
df.loc[mask, 'close'] = df.loc[mask, 'close'] * 100
df.loc[mask, 'volume'] = df.loc[mask, 'volume'] / 100

In [117]:
mask = (df['ticker'] == 'IRAO') & (df['date'] > '2014-12-10')
df[mask].head(20)

,date,open,high,low,close,volume,ticker,close_diff
42295,2014-12-11,0.8367,0.8576,0.7880,0.8070,100564000.0,IRAO,0.034325
42296,2014-12-12,0.8050,0.8195,0.7793,0.7793,77284000.0,IRAO,0.101758
42297,2014-12-15,0.7754,0.7894,0.6863,0.7000,119319000.0,IRAO,0.068571
42298,2014-12-16,0.6899,0.6899,0.5153,0.6520,177598000.0,IRAO,-0.027454
42299,2014-12-17,0.6666,0.6996,0.6015,0.6699,81402000.0,IRAO,-0.062845
42300,2014-12-18,0.6895,0.7200,0.6301,0.7120,110404000.0,IRAO,-106.317416
42309,2015-01-20,0.8394,0.8394,0.7402,0.7641,69485000.0,IRAO,-0.014265
42310,2015-01-21,0.7643,0.8288,0.7600,0.7750,144303000.0,IRAO,-0.020387
42311,2015-01-22,0.7852,0.8068,0.7850,0.7908,43634000.0,IRAO,0.002403
42312,2015-01-23,0.7951,0.8140,0.7810,0.7889,35751000.0,IRAO,0.061985


In [118]:
df['close_diff'] = df.groupby(['ticker'])['close'].diff(-1)/df['close']

In [119]:
df.sort_values(by=['close_diff'])

,date,open,high,low,close,volume,ticker,close_diff
84945,2013-08-14,623.90,623.90,623.900,623.90,2.0,PHOR,-1.198750
134,2014-12-17,5.98,6.90,5.720,6.61,23976100.0,AFKS,-1.062027
118184,2016-02-19,5.05,5.25,5.050,5.16,25700.0,SELG,-0.998062
141419,2022-02-25,277.20,366.40,277.200,296.00,3471329.0,VKCO,-0.722973
64788,2014-12-18,16.90,19.87,16.800,19.35,3550770.0,MTLR,-0.564858
...,...,...,...,...,...,...,...,...
142438,2026-03-06,304.75,305.35,298.350,300.20,1268675.0,VKCO,NaN
142863,2026-03-06,79.85,81.68,78.930,81.09,1460593.0,VSEH,NaN
146134,2026-03-06,85.85,86.68,85.575,86.02,39722553.0,VTBR,NaN
146428,2026-03-06,2414.00,2415.00,2390.500,2395.00,554503.0,X5,NaN


In [128]:
mask = (df['ticker'] == 'PHOR')
df[mask].sort_values(by='date').head(20)

,date,open,high,low,close,volume,ticker,close_diff
84945,2013-08-14,623.9,623.9,623.9,623.9,2.0,PHOR,-1.198750
84946,2013-08-15,621.7,1371.8,621.7,1371.8,30.0,PHOR,0.322132
84958,2013-09-02,995.0,995.0,922.0,929.9,4587.0,PHOR,0.002043
84959,2013-09-03,929.9,935.9,918.1,928.0,17285.0,PHOR,-0.010453
84960,2013-09-04,817.6,944.7,817.6,937.7,10996.0,PHOR,-0.009705
84961,2013-09-05,930.0,947.0,930.0,946.8,4909.0,PHOR,0.002007
84962,2013-09-06,944.1,948.8,930.3,944.9,10736.0,PHOR,0.013652
84963,2013-09-09,931.0,955.0,928.0,932.0,11373.0,PHOR,-0.020386
84964,2013-09-10,938.8,959.9,930.7,951.0,7878.0,PHOR,0.013144
84965,2013-09-11,954.8,958.0,930.0,938.5,7706.0,PHOR,0.003410


In [137]:
df['volume'].describe(np.arange(0,0.1,0.01))

count           144,900.000
mean        195,366,936.820
std       2,955,800,756.230
min                   0.000
0%                    0.000
1%                    0.000
2%                   10.000
3%                  815.000
4%                1,776.960
5%                2,992.850
6%                5,100.000
7%                8,100.000
8%               11,770.000
9%               15,980.910
50%           1,579,920.000
max     661,614,600,000.000
Name: volume, dtype: float64

In [72]:
0.02030*5000

101.5

In [42]:
path = f'prepared_data/dataset_for_feature_engineering.pickle'

with open(path, 'wb') as f:
    pickle.dump(df, f)

In [43]:
df.shape

(144900, 7)